# Recycling VQA Colab (Qwen3-VL + Unsloth + Checkpoints)

이 노트북은 업로드한 baseline을 기준으로 아래를 보강한 버전입니다.

- 모델 preset 선택 가능: 8B / 32B / 직접 HF model id
- LoRA / LR / batch / epoch / save_steps 등 주요 파라미터를 한 곳에서 변경
- step 단위 체크포인트 저장 + 자동 resume
- split 학습과 full-data 2 epoch 학습을 분리
- exact-match 로컬 검증 옵션
- safe augmentation 옵션
- detector crop / OCR crop 같은 추가 이미지 컬럼 지원

권장 순서:
1. **8B preset**으로 split 1 epoch 빠르게 검증
2. 파라미터 확정
3. **full-data 2 epoch**
4. 시간이 충분하면 **32B final run**


## 1) 설치

공식 Unsloth vision notebook 흐름에 맞춰 설치합니다.
- Colab에서 처음 실행하면 5~15분 정도 걸릴 수 있습니다.
- 설치 후 경고가 조금 떠도 진행 가능한 경우가 많습니다.


In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip -q install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip -q install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip -q install transformers==4.57.1 trl==0.22.2 pandas pillow scikit-learn


## 2) 드라이브 마운트 / 데이터 압축 해제

baseline과 비슷하게 Google Drive에서 zip을 풀 수 있게 해두었습니다.
이미 `/content/train.csv`, `/content/test.csv`가 준비되어 있으면 이 셀은 건너뛰어도 됩니다.


In [ ]:
# 필요할 때만 실행
USE_DRIVE = True
DRIVE_ZIP_PATH = "/content/drive/MyDrive/SSAFY15/AI_challenge_260402/2026-ssafy-15-2-ai_dataset.zip"
UNZIP_DIR = "/content"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    if os.path.exists(DRIVE_ZIP_PATH):
        !unzip -o "{DRIVE_ZIP_PATH}" -d "{UNZIP_DIR}"
    else:
        print(f"[WARN] zip not found: {DRIVE_ZIP_PATH}")
else:
    print("Drive mount skipped.")


## 3) 공통 설정

여기만 바꾸면 됩니다.
- 빠른 시작: `MODEL_PRESET = "qwen3_vl_8b_unsloth_dynamic"`
- 최종 점수용: `MODEL_PRESET = "qwen3_vl_32b_bnb4"`
- detector crop을 쓰면 `AUX_IMAGE_COLS`에 컬럼명 추가


In [ ]:
from pathlib import Path
import os
import pandas as pd
import torch

# -------------------------
# 데이터 경로
# -------------------------
TRAIN_CSV = "/content/train.csv"
TEST_CSV  = "/content/test.csv"
VALID_CSV = None              # 별도 valid.csv가 있으면 경로 입력
EXTRA_TRAIN_CSV = None        # TACO/COCO relabel 등 추가 데이터가 있으면 경로 입력
IMAGE_ROOT = "/content"       # path 컬럼이 상대경로면 이 루트 기준으로 찾음

# -------------------------
# 모델 선택
# -------------------------
MODEL_PRESET = "qwen3_vl_8b_unsloth_dynamic"
# MODEL_PRESET = "qwen3_vl_32b_bnb4"
# MODEL_PRESET = "qwen3_vl_8b_bnb4"
# MODEL_PRESET = "qwen3_vl_4b_debug"

CUSTOM_MODEL_NAME = None      # 직접 HF model id를 넣고 싶으면 문자열로 설정

# -------------------------
# 실험 이름 / 출력 경로
# -------------------------
EXP_NAME = "recycling_qwen3vl_exp01"
OUTPUT_ROOT = "/content/outputs"
SPLIT_OUTPUT_DIR = f"{OUTPUT_ROOT}/{EXP_NAME}_split"
FULL_OUTPUT_DIR  = f"{OUTPUT_ROOT}/{EXP_NAME}_full2ep"

# -------------------------
# 데이터 / 컬럼 설정
# -------------------------
IMAGE_COL = "path"
QUESTION_COL = "question"
OPTION_COLS = ["a", "b", "c", "d"]
ANSWER_COL = "answer"
CATEGORY_COL = "category"     # 없으면 무시됨

# detector crop / OCR crop / count crop 등을 추가할 때 사용
AUX_IMAGE_COLS = []           # 예: ["roi_path", "ocr_crop_path"]
DETECTOR_HINT_COL = None      # 예: "detector_hint"

# -------------------------
# 학습 전략
# -------------------------
VALID_RATIO = 0.1
SEARCH_EPOCHS = 1.0
FULL_TRAIN_EPOCHS = 2.0       # 이전 기수 방식 반영
DEBUG_SUBSET = None           # 예: 300 으로 두면 smoke test
REMOVE_IDENTICAL_OPTION_ROWS = True
ENABLE_AUGMENTATION = False   # safe augmentation만 적용

# -------------------------
# LoRA / 최적화 파라미터
# None이면 preset 기본값 사용
# -------------------------
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LEARNING_RATE = None
TRAIN_BATCH_SIZE = None
GRAD_ACCUM = None
MAX_SEQ_LENGTH = 2048

# -------------------------
# 체크포인트 / 로그
# -------------------------
SAVE_STEPS = 100
EVAL_STEPS = 100
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 3
AUTO_RESUME = True
RUN_EXACT_MATCH_EVAL = True
EVAL_MAX_SAMPLES = 200

# -------------------------
# 예측 설정
# -------------------------
PREDICT_FROM = "full"         # "split" 또는 "full"
PREDICTION_OUTPUT = "/content/submission.csv"
PRED_MAX_NEW_TOKENS = 4

# -------------------------
# 재현성
# -------------------------
SEED = 3407

print("Torch:", torch.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU :", props.name)
    print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")
else:
    print("CUDA not available")


## 4) 디버그 subset 준비 (선택)

처음에는 200~500개 정도로 파이프라인부터 확인하는 걸 권장합니다.
`DEBUG_SUBSET = None`이면 원본 train.csv를 그대로 사용합니다.


In [ ]:
import pandas as pd
from pathlib import Path

train_csv_for_run = TRAIN_CSV
valid_csv_for_run = VALID_CSV

if DEBUG_SUBSET is not None:
    df = pd.read_csv(TRAIN_CSV)
    n = min(int(DEBUG_SUBSET), len(df))
    df = df.sample(n=n, random_state=SEED).reset_index(drop=True)

    debug_dir = Path("/content/debug_inputs")
    debug_dir.mkdir(parents=True, exist_ok=True)
    train_csv_for_run = str(debug_dir / f"train_debug_{n}.csv")
    df.to_csv(train_csv_for_run, index=False)

    print(f"[INFO] Debug subset saved: {train_csv_for_run} ({len(df)} rows)")
else:
    print(f"[INFO] Using full train CSV: {train_csv_for_run}")

assert Path(train_csv_for_run).exists(), f"Missing train csv: {train_csv_for_run}"
assert Path(TEST_CSV).exists(), f"Missing test csv: {TEST_CSV}"
print("[OK] CSV paths ready.")


## 5) 학습 스크립트 저장

아래 셀은 이번에 수정한 학습 스크립트를 `/content/recycling_vqa_colab_train_v2.py`로 저장합니다.


In [ ]:
%%writefile /content/recycling_vqa_colab_train_v2.py
#!/usr/bin/env python3
"""
Colab-ready Recycling VQA training / inference script.

What this script is for
-----------------------
- Fine-tune a Qwen3-VL family model with Unsloth + TRL on a recycling / waste / litter VQA dataset.
- Switch models with presets or a direct Hugging Face model id.
- Change LoRA / optimizer / augmentation / checkpoint parameters from CLI.
- Save checkpoints during training and automatically resume after Colab interruptions.
- Support optional multi-image training for detector-assisted ROI crops.
- Run prediction and build a submission CSV.

Recommended workflow
--------------------
1) Quick smoke test on 100-300 rows with 8B.
2) Proper split training (train/valid) for hyperparameter search.
3) Full-data training with the best setting for 2 epochs.
4) Optional second run with 32B on A100/H100.
5) Optional detector-assisted crops via extra image columns.

Expected CSV schema (default)
-----------------------------
train.csv / valid.csv:
    id, path, question, a, b, c, d, answer

You can override all column names from CLI.
`answer` can be either:
    - one of: a / b / c / d
    - or the exact option text (the script maps it back to a/b/c/d)

Optional extra columns:
    - category
    - roi_path / ocr_crop_path / count_crop_path ...
      (any image columns you pass in --aux_image_cols)

Important note
--------------
This is a fresh, configurable training script. If you upload your exact baseline code,
I can patch it line-by-line too. Right now this script is designed to be dropped into Colab
and run immediately.
"""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import re
import shutil
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

import torch
from torch.utils.data import Dataset


# -----------------------------
# Model presets
# -----------------------------
MODEL_PRESETS: Dict[str, Dict[str, Any]] = {
    # Fast debug model.
    "qwen3_vl_4b_debug": {
        "model_name": "Qwen/Qwen3-VL-4B-Instruct",
        "load_in_4bit": False,
        "max_seq_length": 2048,
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 1.0,
        "notes": "Debug / smoke-test preset. Use first when checking data pipeline.",
    },
    # Best quick-start candidate for most A100/H100 fine-tuning runs.
    "qwen3_vl_8b_unsloth_dynamic": {
        "model_name": "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
        "load_in_4bit": True,
        "max_seq_length": 2048,
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 1.0,
        "notes": "Strong default for quick start. Good fit for A100/H100 and previous cohort experience.",
    },
    # Slightly more conservative 8B quantized option.
    "qwen3_vl_8b_bnb4": {
        "model_name": "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit",
        "load_in_4bit": True,
        "max_seq_length": 2048,
        "learning_rate": 1.5e-4,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 1.0,
        "notes": "Conservative 8B preset when you want a simpler 4-bit path.",
    },
    # Large-model final-run candidate.
    "qwen3_vl_32b_bnb4": {
        "model_name": "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
        "load_in_4bit": True,
        "max_seq_length": 2048,
        "learning_rate": 1e-4,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "num_train_epochs": 1.0,
        "notes": "Recommended large final-run preset when 32B fits. Prefer this over the dynamic 32B if memory is tight.",
    },
    # Inference-only hard-case reranker, not the default fine-tuning target.
    "qwen3_vl_32b_thinking": {
        "model_name": "Qwen/Qwen3-VL-32B-Thinking",
        "load_in_4bit": False,
        "max_seq_length": 2048,
        "learning_rate": 5e-5,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "num_train_epochs": 1.0,
        "notes": "Use mainly for hard-case inference / reranking, not as the first fine-tuning run.",
    },
}


# -----------------------------
# Utility helpers
# -----------------------------
ANSWER_LETTERS = ["a", "b", "c", "d"]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def is_colab() -> bool:
    return "google.colab" in sys.modules


def latest_checkpoint(output_dir: str | Path) -> Optional[str]:
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return None
    candidates = []
    for p in output_dir.glob("checkpoint-*"):
        m = re.search(r"checkpoint-(\d+)$", p.name)
        if m:
            candidates.append((int(m.group(1)), p))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return str(candidates[-1][1])


def first_existing_path(path_value: Any, image_root: str | Path) -> Optional[Path]:
    if path_value is None:
        return None
    path_str = str(path_value).strip()
    if not path_str or path_str.lower() == "nan":
        return None
    p = Path(path_str)
    if p.exists():
        return p
    p2 = Path(image_root) / path_str
    if p2.exists():
        return p2
    return None


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_label(raw_answer: Any, options: Dict[str, str]) -> Optional[str]:
    if raw_answer is None:
        return None
    ans = normalize_text(raw_answer)
    if ans in ANSWER_LETTERS:
        return ans

    # Allow answers like "A", "b.", "정답: c"
    m = re.search(r"\b([abcd])\b", ans)
    if m:
        return m.group(1)

    for letter, option_text in options.items():
        if ans == normalize_text(option_text):
            return letter

    # Sometimes answer is a fragment that exactly equals one option after cleanup.
    for letter, option_text in options.items():
        if ans and ans in normalize_text(option_text):
            return letter

    return None


def parse_prediction(raw_text: str, options: Dict[str, str]) -> str:
    text = normalize_text(raw_text)
    if text in ANSWER_LETTERS:
        return text

    m = re.search(r"\b([abcd])\b", text)
    if m:
        return m.group(1)

    if text and text[0] in ANSWER_LETTERS:
        return text[0]

    for letter, option_text in options.items():
        opt = normalize_text(option_text)
        if text == opt:
            return letter
        if opt and opt in text:
            return letter

    # Safe fallback so submission format never breaks.
    return "a"


def infer_question_type(question: str, category: Optional[str] = None) -> str:
    q = normalize_text(question)
    cat = normalize_text(category)
    merged = f"{q} {cat}".strip()

    position_keywords = ["왼쪽", "오른쪽", "위", "아래", "앞", "뒤", "옆", "가까이", "멀리", "위치", "방향"]
    ocr_keywords = ["글자", "문자", "적힌", "문구", "간판", "브랜드", "상표", "이름", "무슨 글", "text", "ocr"]
    color_keywords = ["무슨 색", "색", "색깔", "컬러", "빨간", "파란", "초록", "노란", "검은", "하얀"]
    count_keywords = ["몇 개", "몇개", "개수", "몇 명", "몇명", "count", "갯수"]

    if any(k in merged for k in position_keywords):
        return "position"
    if any(k in merged for k in ocr_keywords):
        return "ocr"
    if any(k in merged for k in color_keywords):
        return "color"
    if any(k in merged for k in count_keywords):
        return "count"
    return "general"


# -----------------------------
# Image loading / safe augmentation
# -----------------------------

def load_image(path: Path, resize_long_edge: int = 960) -> Image.Image:
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert("RGB")

    if resize_long_edge and resize_long_edge > 0:
        w, h = img.size
        long_edge = max(w, h)
        if long_edge > resize_long_edge:
            scale = resize_long_edge / float(long_edge)
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))
            img = img.resize((new_w, new_h), Image.BICUBIC)
    return img


@dataclass
class AugConfig:
    enabled: bool = False
    p_brightness: float = 0.25
    p_contrast: float = 0.25
    p_saturation: float = 0.20
    p_sharpness: float = 0.20
    p_noise: float = 0.15
    brightness_delta: float = 0.10
    contrast_delta: float = 0.10
    saturation_delta: float = 0.05
    sharpness_delta: float = 0.10
    noise_std: float = 4.0


class SafeVQAAugmenter:
    def __init__(self, cfg: AugConfig):
        self.cfg = cfg

    def __call__(self, image: Image.Image, question_type: str) -> Image.Image:
        if not self.cfg.enabled:
            return image

        img = image.copy()

        # Never use flips / rotations / strong crops / hue shift here.
        # Question-type gating is deliberately conservative.
        allow_saturation = question_type not in {"color"}
        allow_noise = question_type not in {"ocr", "count"}
        allow_sharpness = question_type not in {"count"}

        if random.random() < self.cfg.p_brightness:
            factor = 1.0 + random.uniform(-self.cfg.brightness_delta, self.cfg.brightness_delta)
            img = ImageEnhance.Brightness(img).enhance(factor)

        if random.random() < self.cfg.p_contrast:
            factor = 1.0 + random.uniform(-self.cfg.contrast_delta, self.cfg.contrast_delta)
            img = ImageEnhance.Contrast(img).enhance(factor)

        if allow_saturation and random.random() < self.cfg.p_saturation:
            factor = 1.0 + random.uniform(-self.cfg.saturation_delta, self.cfg.saturation_delta)
            img = ImageEnhance.Color(img).enhance(factor)

        if allow_sharpness and random.random() < self.cfg.p_sharpness:
            factor = 1.0 + random.uniform(-self.cfg.sharpness_delta, self.cfg.sharpness_delta)
            img = ImageEnhance.Sharpness(img).enhance(factor)

        if allow_noise and random.random() < self.cfg.p_noise:
            arr = np.array(img).astype(np.float32)
            arr += np.random.normal(0.0, self.cfg.noise_std, arr.shape)
            arr = np.clip(arr, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)

        return img


# -----------------------------
# Dataset
# -----------------------------

def build_prompt(
    question: str,
    options: Dict[str, str],
    detector_hint: Optional[str] = None,
) -> str:
    base = [
        "이미지를 보고 질문에 답하세요.",
        f"질문: {question}",
        "선택지:",
        f"a. {options['a']}",
        f"b. {options['b']}",
        f"c. {options['c']}",
        f"d. {options['d']}",
    ]
    if detector_hint:
        base.append(f"참고 정보(자동 탐지): {detector_hint}")
    base.extend(
        [
            "규칙:",
            "- 반드시 a, b, c, d 중 하나만 출력하세요.",
            "- 설명하지 마세요.",
        ]
    )
    return "\n".join(base)


class RecyclingVQADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_root: str | Path,
        image_col: str,
        question_col: str,
        option_cols: Sequence[str],
        answer_col: Optional[str],
        category_col: Optional[str],
        aux_image_cols: Sequence[str],
        detector_hint_col: Optional[str],
        resize_long_edge: int,
        augmenter: Optional[SafeVQAAugmenter],
        is_train: bool,
        system_prompt: str,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.image_col = image_col
        self.question_col = question_col
        self.option_cols = list(option_cols)
        self.answer_col = answer_col
        self.category_col = category_col
        self.aux_image_cols = list(aux_image_cols)
        self.detector_hint_col = detector_hint_col
        self.resize_long_edge = resize_long_edge
        self.augmenter = augmenter
        self.is_train = is_train
        self.system_prompt = system_prompt

    def __len__(self) -> int:
        return len(self.df)

    def _load_images(self, row: pd.Series, qtype: str) -> List[Image.Image]:
        main_path = first_existing_path(row[self.image_col], self.image_root)
        if main_path is None:
            raise FileNotFoundError(f"Main image not found: {row[self.image_col]}")

        images = [load_image(main_path, resize_long_edge=self.resize_long_edge)]

        for col in self.aux_image_cols:
            if col not in row.index:
                continue
            extra_path = first_existing_path(row[col], self.image_root)
            if extra_path is not None:
                images.append(load_image(extra_path, resize_long_edge=self.resize_long_edge))

        if self.is_train and self.augmenter is not None:
            images = [self.augmenter(img, qtype) for img in images]

        return images

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        question = str(row[self.question_col])
        category = str(row[self.category_col]) if (self.category_col and self.category_col in row.index) else ""
        qtype = infer_question_type(question, category)

        options = {
            "a": str(row[self.option_cols[0]]),
            "b": str(row[self.option_cols[1]]),
            "c": str(row[self.option_cols[2]]),
            "d": str(row[self.option_cols[3]]),
        }

        detector_hint = None
        if self.detector_hint_col and self.detector_hint_col in row.index:
            raw_hint = row[self.detector_hint_col]
            if pd.notna(raw_hint):
                detector_hint = str(raw_hint)

        images = self._load_images(row, qtype)
        prompt = build_prompt(question=question, options=options, detector_hint=detector_hint)

        user_contents: List[Dict[str, Any]] = []
        for img in images:
            user_contents.append({"type": "image", "image": img})
        user_contents.append({"type": "text", "text": prompt})

        messages: List[Dict[str, Any]] = [
            {"role": "system", "content": [{"type": "text", "text": self.system_prompt}]},
            {"role": "user", "content": user_contents},
        ]

        gold_letter: Optional[str] = None
        if self.answer_col and self.answer_col in row.index and pd.notna(row[self.answer_col]):
            gold_letter = normalize_label(row[self.answer_col], options)
            if gold_letter is None:
                raise ValueError(f"Could not normalize answer for row idx={idx}: {row[self.answer_col]}")
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold_letter}]})

        item = {
            "messages": messages,
            "meta": {
                "id": str(row["id"]) if "id" in row.index else str(idx),
                "question_type": qtype,
                "options": options,
                "gold_letter": gold_letter,
                "question": question,
            },
        }
        return item


# -----------------------------
# Dataframe preparation
# -----------------------------

def load_dataframe(csv_path: str | Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    return pd.read_csv(csv_path)


def check_required_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c and c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {name}: {missing}")


def remove_identical_option_rows(df: pd.DataFrame, option_cols: Sequence[str]) -> Tuple[pd.DataFrame, int]:
    def has_duplicate_options(row: pd.Series) -> bool:
        vals = [normalize_text(row[c]) for c in option_cols]
        return len(set(vals)) < len(vals)

    mask = df.apply(has_duplicate_options, axis=1)
    removed = int(mask.sum())
    return df.loc[~mask].reset_index(drop=True), removed


def filter_missing_images(df: pd.DataFrame, image_root: str | Path, image_col: str, aux_image_cols: Sequence[str]) -> Tuple[pd.DataFrame, int]:
    keep_rows = []
    dropped = 0
    for _, row in df.iterrows():
        main_path = first_existing_path(row[image_col], image_root)
        if main_path is None:
            dropped += 1
            continue
        keep_rows.append(row)
    return pd.DataFrame(keep_rows).reset_index(drop=True), dropped


def train_valid_split(
    df: pd.DataFrame,
    valid_ratio: float,
    seed: int,
    stratify_col: Optional[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if valid_ratio <= 0.0:
        return df.reset_index(drop=True), pd.DataFrame(columns=df.columns)

    from sklearn.model_selection import train_test_split

    stratify_values = None
    if stratify_col and stratify_col in df.columns:
        vc = df[stratify_col].value_counts(dropna=False)
        if (vc >= 2).all():
            stratify_values = df[stratify_col]

    train_df, valid_df = train_test_split(
        df,
        test_size=valid_ratio,
        random_state=seed,
        stratify=stratify_values,
    )
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True)


# -----------------------------
# Training / evaluation / inference
# -----------------------------

def load_unsloth_model(
    model_name: str,
    load_in_4bit: bool,
    max_seq_length: int,
) -> Tuple[Any, Any]:
    from unsloth import FastVisionModel

    dtype = None  # let Unsloth auto-select fp16/bf16 sensibly
    model, processor = FastVisionModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
        use_gradient_checkpointing="unsloth",
    )
    return model, processor


def attach_lora(
    model: Any,
    r: int,
    lora_alpha: int,
    lora_dropout: float,
    bias: str,
    random_state: int,
    finetune_vision_layers: bool,
    finetune_language_layers: bool,
    finetune_attention_modules: bool,
    finetune_mlp_modules: bool,
) -> Any:
    from unsloth import FastVisionModel

    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=finetune_vision_layers,
        finetune_language_layers=finetune_language_layers,
        finetune_attention_modules=finetune_attention_modules,
        finetune_mlp_modules=finetune_mlp_modules,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias=bias,
        random_state=random_state,
        use_rslora=False,
        loftq_config=None,
        target_modules="all-linear",
        modules_to_save=["lm_head", "embed_tokens"],
    )
    return model


def maybe_load_adapter(model: Any, adapter_path: Optional[str]) -> Any:
    if not adapter_path:
        return model
    from peft import PeftModel

    return PeftModel.from_pretrained(model, adapter_path, is_trainable=False)


def make_sft_config(args: argparse.Namespace, has_eval: bool) -> Any:
    from trl import SFTConfig

    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not bf16

    sft_kwargs = dict(
        output_dir=args.output_dir,
        num_train_epochs=args.num_train_epochs,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        learning_rate=args.learning_rate,
        warmup_ratio=args.warmup_ratio,
        lr_scheduler_type=args.lr_scheduler_type,
        weight_decay=args.weight_decay,
        optim=args.optim,
        logging_steps=args.logging_steps,
        save_strategy="steps",
        save_steps=args.save_steps,
        save_total_limit=args.save_total_limit,
        max_length=None,
        bf16=bf16,
        fp16=fp16,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        gradient_checkpointing=False,  # Unsloth already patches this at model load time.
        report_to=args.report_to if args.report_to != "none" else None,
        dataloader_num_workers=args.dataloader_num_workers,
        eos_token="<|im_end|>",
        seed=args.seed,
    )

    if has_eval:
        sft_kwargs.update(
            dict(
                do_eval=True,
                eval_strategy="steps",
                eval_steps=args.eval_steps,
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
                greater_is_better=False,
            )
        )
    else:
        sft_kwargs.update(dict(do_eval=False, eval_strategy="no", load_best_model_at_end=False))

    return SFTConfig(**sft_kwargs)


def generate_one(
    model: Any,
    processor: Any,
    messages: List[Dict[str, Any]],
    max_new_tokens: int = 4,
) -> str:
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    input_len = inputs["input_ids"].shape[-1]
    trimmed = generated_ids[:, input_len:]
    text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return text.strip()


def evaluate_exact_match(
    model: Any,
    processor: Any,
    dataset: Dataset,
    max_samples: Optional[int] = None,
) -> Dict[str, Any]:
    from unsloth import FastVisionModel

    FastVisionModel.for_inference(model)
    total = 0
    correct = 0
    by_type: Dict[str, List[int]] = {}

    limit = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for i in range(limit):
        item = dataset[i]
        messages = item["messages"][:-1] if item["meta"]["gold_letter"] is not None else item["messages"]
        raw_pred = generate_one(model, processor, messages)
        pred = parse_prediction(raw_pred, item["meta"]["options"])
        gold = item["meta"]["gold_letter"]
        qtype = item["meta"]["question_type"]
        ok = int(pred == gold)

        total += 1
        correct += ok
        by_type.setdefault(qtype, []).append(ok)

    result = {
        "exact_match": (correct / total) if total else 0.0,
        "num_samples": total,
        "by_type": {k: float(np.mean(v)) for k, v in sorted(by_type.items())},
    }
    return result


def train(args: argparse.Namespace) -> None:
    seed_everything(args.seed)
    ensure_dir(args.output_dir)

    train_df = load_dataframe(args.train_csv)
    valid_df = load_dataframe(args.valid_csv) if args.valid_csv else None
    extra_df = load_dataframe(args.extra_train_csv) if args.extra_train_csv else None

    required_train_cols = [args.image_col, args.question_col, *args.option_cols]
    if args.answer_col:
        required_train_cols.append(args.answer_col)
    check_required_columns(train_df, required_train_cols, "train_csv")
    if valid_df is not None:
        check_required_columns(valid_df, required_train_cols, "valid_csv")
    if extra_df is not None:
        check_required_columns(extra_df, required_train_cols, "extra_train_csv")

    if args.remove_identical_option_rows:
        train_df, removed = remove_identical_option_rows(train_df, args.option_cols)
        print(f"[INFO] Removed {removed} train rows with duplicated options.")
        if valid_df is not None and len(valid_df) > 0:
            valid_df, v_removed = remove_identical_option_rows(valid_df, args.option_cols)
            print(f"[INFO] Removed {v_removed} valid rows with duplicated options.")
        if extra_df is not None and len(extra_df) > 0:
            extra_df, e_removed = remove_identical_option_rows(extra_df, args.option_cols)
            print(f"[INFO] Removed {e_removed} extra rows with duplicated options.")

    train_df, dropped_train = filter_missing_images(train_df, args.image_root, args.image_col, args.aux_image_cols)
    print(f"[INFO] Dropped {dropped_train} train rows with missing main image.")

    if valid_df is None and args.valid_ratio > 0:
        stratify_col = args.category_col if args.category_col and args.category_col in train_df.columns else None
        train_df, valid_df = train_valid_split(train_df, args.valid_ratio, args.seed, stratify_col)
        print(f"[INFO] Split train/valid -> {len(train_df)} / {len(valid_df)}")
    elif valid_df is not None:
        valid_df, dropped_valid = filter_missing_images(valid_df, args.image_root, args.image_col, args.aux_image_cols)
        print(f"[INFO] Dropped {dropped_valid} valid rows with missing main image.")

    if extra_df is not None and len(extra_df) > 0:
        extra_df, dropped_extra = filter_missing_images(extra_df, args.image_root, args.image_col, args.aux_image_cols)
        print(f"[INFO] Dropped {dropped_extra} extra rows with missing main image.")
        train_df = pd.concat([train_df, extra_df], ignore_index=True)
        print(f"[INFO] Appended extra data. New train size: {len(train_df)}")

    aug_cfg = AugConfig(enabled=args.enable_augmentation)
    augmenter = SafeVQAAugmenter(aug_cfg) if args.enable_augmentation else None

    train_dataset = RecyclingVQADataset(
        df=train_df,
        image_root=args.image_root,
        image_col=args.image_col,
        question_col=args.question_col,
        option_cols=args.option_cols,
        answer_col=args.answer_col,
        category_col=args.category_col,
        aux_image_cols=args.aux_image_cols,
        detector_hint_col=args.detector_hint_col,
        resize_long_edge=args.resize_long_edge,
        augmenter=augmenter,
        is_train=True,
        system_prompt=args.system_prompt,
    )

    eval_dataset = None
    if valid_df is not None and len(valid_df) > 0:
        eval_dataset = RecyclingVQADataset(
            df=valid_df,
            image_root=args.image_root,
            image_col=args.image_col,
            question_col=args.question_col,
            option_cols=args.option_cols,
            answer_col=args.answer_col,
            category_col=args.category_col,
            aux_image_cols=args.aux_image_cols,
            detector_hint_col=args.detector_hint_col,
            resize_long_edge=args.resize_long_edge,
            augmenter=None,
            is_train=False,
            system_prompt=args.system_prompt,
        )

    model, processor = load_unsloth_model(
        model_name=args.model_name,
        load_in_4bit=args.load_in_4bit,
        max_seq_length=args.max_seq_length,
    )
    model = attach_lora(
        model=model,
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias=args.lora_bias,
        random_state=args.seed,
        finetune_vision_layers=args.finetune_vision_layers,
        finetune_language_layers=args.finetune_language_layers,
        finetune_attention_modules=args.finetune_attention_modules,
        finetune_mlp_modules=args.finetune_mlp_modules,
    )

    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()

    from unsloth import FastVisionModel
    from unsloth.trainer import UnslothVisionDataCollator
    from trl import SFTTrainer

    FastVisionModel.for_training(model)

    collator = UnslothVisionDataCollator(
        model,
        processor,
        resize="min",
        train_on_responses_only=args.train_on_responses_only,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
        completion_only_loss=True,
    )

    sft_config = make_sft_config(args, has_eval=eval_dataset is not None)

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        data_collator=collator,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=processor,
    )

    # Save the exact run config for reproducibility.
    with open(Path(args.output_dir) / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(vars(args), f, ensure_ascii=False, indent=2)

    resume_ckpt = args.resume_from_checkpoint
    if args.auto_resume and not resume_ckpt:
        resume_ckpt = latest_checkpoint(args.output_dir)
        if resume_ckpt:
            print(f"[INFO] Auto-resuming from {resume_ckpt}")

    train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
    trainer.save_state()

    final_dir = ensure_dir(Path(args.output_dir) / "final_adapter")
    trainer.model.save_pretrained(final_dir)
    processor.save_pretrained(final_dir)

    with open(Path(args.output_dir) / "train_metrics.json", "w", encoding="utf-8") as f:
        json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

    # Optional generation-based exact-match evaluation.
    if eval_dataset is not None and args.run_exact_match_eval:
        em = evaluate_exact_match(model=trainer.model, processor=processor, dataset=eval_dataset, max_samples=args.eval_max_samples)
        with open(Path(args.output_dir) / "exact_match_eval.json", "w", encoding="utf-8") as f:
            json.dump(em, f, ensure_ascii=False, indent=2)
        print("[INFO] Exact-match evaluation:")
        print(json.dumps(em, ensure_ascii=False, indent=2))

    print(f"[DONE] Training finished. Final adapter saved to: {final_dir}")


def predict(args: argparse.Namespace) -> None:
    seed_everything(args.seed)
    test_df = load_dataframe(args.test_csv)
    check_required_columns(test_df, [args.image_col, args.question_col, *args.option_cols], "test_csv")
    test_df, dropped = filter_missing_images(test_df, args.image_root, args.image_col, args.aux_image_cols)
    print(f"[INFO] Dropped {dropped} test rows with missing main image.")

    dataset = RecyclingVQADataset(
        df=test_df,
        image_root=args.image_root,
        image_col=args.image_col,
        question_col=args.question_col,
        option_cols=args.option_cols,
        answer_col=None,
        category_col=args.category_col,
        aux_image_cols=args.aux_image_cols,
        detector_hint_col=args.detector_hint_col,
        resize_long_edge=args.resize_long_edge,
        augmenter=None,
        is_train=False,
        system_prompt=args.system_prompt,
    )

    model, processor = load_unsloth_model(
        model_name=args.model_name,
        load_in_4bit=args.load_in_4bit,
        max_seq_length=args.max_seq_length,
    )
    model = maybe_load_adapter(model, args.adapter_path)

    from unsloth import FastVisionModel

    FastVisionModel.for_inference(model)

    preds = []
    raw_texts = []
    ids = []

    for i in range(len(dataset)):
        item = dataset[i]
        raw_pred = generate_one(model, processor, item["messages"], max_new_tokens=args.pred_max_new_tokens)
        pred = parse_prediction(raw_pred, item["meta"]["options"])
        raw_texts.append(raw_pred)
        preds.append(pred)
        ids.append(item["meta"]["id"])
        if (i + 1) % max(1, args.pred_log_every) == 0:
            print(f"[PRED] {i+1}/{len(dataset)}")

    out_df = pd.DataFrame({
        args.id_output_col: ids,
        args.pred_output_col: preds,
        "raw_prediction": raw_texts,
    })
    out_path = Path(args.prediction_output)
    ensure_dir(out_path.parent)
    out_df.to_csv(out_path, index=False)
    print(f"[DONE] Saved predictions to: {out_path}")


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Recycling VQA fine-tuning / inference with Unsloth + Qwen3-VL")

    # Execution mode.
    parser.add_argument("--mode", choices=["train", "predict"], default="train")

    # Model selection.
    parser.add_argument("--preset", type=str, default="qwen3_vl_8b_unsloth_dynamic", choices=list(MODEL_PRESETS.keys()))
    parser.add_argument("--model_name", type=str, default=None, help="Override model preset with an explicit HF model id.")
    parser.add_argument("--load_in_4bit", type=lambda x: str(x).lower() == "true", default=None)
    parser.add_argument("--max_seq_length", type=int, default=None)

    # Data paths.
    parser.add_argument("--image_root", type=str, default=".")
    parser.add_argument("--train_csv", type=str, default=None)
    parser.add_argument("--valid_csv", type=str, default=None)
    parser.add_argument("--extra_train_csv", type=str, default=None)
    parser.add_argument("--test_csv", type=str, default=None)

    # Column names.
    parser.add_argument("--image_col", type=str, default="path")
    parser.add_argument("--question_col", type=str, default="question")
    parser.add_argument("--option_cols", nargs=4, default=["a", "b", "c", "d"])
    parser.add_argument("--answer_col", type=str, default="answer")
    parser.add_argument("--category_col", type=str, default="category")
    parser.add_argument("--aux_image_cols", nargs="*", default=[])
    parser.add_argument("--detector_hint_col", type=str, default=None)

    # Train / split behavior.
    parser.add_argument("--valid_ratio", type=float, default=0.1)
    parser.add_argument("--remove_identical_option_rows", action="store_true")
    parser.add_argument("--seed", type=int, default=3407)

    # Prompt / preprocessing.
    parser.add_argument("--resize_long_edge", type=int, default=960)
    parser.add_argument(
        "--system_prompt",
        type=str,
        default="당신은 재활용과 폐기물 장면에 강한 시각적 질의응답 전문가입니다. 항상 정확하게 판단하고, 정답은 a, b, c, d 중 하나만 출력합니다.",
    )

    # Augmentation.
    parser.add_argument("--enable_augmentation", action="store_true")

    # LoRA.
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=16)
    parser.add_argument("--lora_dropout", type=float, default=0.0)
    parser.add_argument("--lora_bias", type=str, default="none")
    parser.add_argument("--finetune_vision_layers", type=lambda x: str(x).lower() == "true", default=True)
    parser.add_argument("--finetune_language_layers", type=lambda x: str(x).lower() == "true", default=True)
    parser.add_argument("--finetune_attention_modules", type=lambda x: str(x).lower() == "true", default=True)
    parser.add_argument("--finetune_mlp_modules", type=lambda x: str(x).lower() == "true", default=True)
    parser.add_argument("--train_on_responses_only", type=lambda x: str(x).lower() == "true", default=True)

    # Training hyperparameters.
    parser.add_argument("--output_dir", type=str, default="./outputs/recycling_vqa_run")
    parser.add_argument("--num_train_epochs", type=float, default=None)
    parser.add_argument("--learning_rate", type=float, default=None)
    parser.add_argument("--warmup_ratio", type=float, default=0.03)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--lr_scheduler_type", type=str, default="cosine")
    parser.add_argument("--optim", type=str, default="adamw_8bit")
    parser.add_argument("--per_device_train_batch_size", type=int, default=None)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=1)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=None)
    parser.add_argument("--dataloader_num_workers", type=int, default=2)

    # Logging / checkpointing.
    parser.add_argument("--report_to", type=str, default="none")  # wandb / tensorboard / none
    parser.add_argument("--logging_steps", type=int, default=10)
    parser.add_argument("--save_steps", type=int, default=100)
    parser.add_argument("--eval_steps", type=int, default=100)
    parser.add_argument("--save_total_limit", type=int, default=3)
    parser.add_argument("--resume_from_checkpoint", type=str, default=None)
    parser.add_argument("--auto_resume", action="store_true")

    # Validation.
    parser.add_argument("--run_exact_match_eval", action="store_true")
    parser.add_argument("--eval_max_samples", type=int, default=None)

    # Prediction.
    parser.add_argument("--adapter_path", type=str, default=None)
    parser.add_argument("--prediction_output", type=str, default="./outputs/predictions.csv")
    parser.add_argument("--pred_output_col", type=str, default="answer")
    parser.add_argument("--id_output_col", type=str, default="id")
    parser.add_argument("--pred_max_new_tokens", type=int, default=4)
    parser.add_argument("--pred_log_every", type=int, default=100)

    return parser


def apply_preset_defaults(args: argparse.Namespace) -> argparse.Namespace:
    preset = MODEL_PRESETS[args.preset]
    if args.model_name is None:
        args.model_name = preset["model_name"]
    if args.load_in_4bit is None:
        args.load_in_4bit = bool(preset["load_in_4bit"])
    if args.max_seq_length is None:
        args.max_seq_length = int(preset["max_seq_length"])
    if args.learning_rate is None:
        args.learning_rate = float(preset["learning_rate"])
    if args.per_device_train_batch_size is None:
        args.per_device_train_batch_size = int(preset["per_device_train_batch_size"])
    if args.gradient_accumulation_steps is None:
        args.gradient_accumulation_steps = int(preset["gradient_accumulation_steps"])
    if args.num_train_epochs is None:
        args.num_train_epochs = float(preset["num_train_epochs"])
    return args


def validate_mode_specific_args(args: argparse.Namespace) -> None:
    if args.mode == "train":
        if not args.train_csv:
            raise ValueError("--train_csv is required for train mode.")
    elif args.mode == "predict":
        if not args.test_csv:
            raise ValueError("--test_csv is required for predict mode.")


def main() -> None:
    parser = build_arg_parser()
    args = parser.parse_args()
    args = apply_preset_defaults(args)
    validate_mode_specific_args(args)

    print("[CONFIG]")
    print(json.dumps(vars(args), ensure_ascii=False, indent=2))
    print(f"[PRESET_NOTE] {MODEL_PRESETS[args.preset]['notes']}")

    if args.mode == "train":
        train(args)
    elif args.mode == "predict":
        predict(args)
    else:
        raise ValueError(f"Unknown mode: {args.mode}")


if __name__ == "__main__":
    main()


## 6) 1차 split 학습

이 단계는 **모델/파라미터 탐색용**입니다.
- 기본 추천: 8B, 1 epoch
- valid_ratio=0.1
- 체크포인트는 `checkpoint-*` 형식으로 저장됩니다.
- 중간에 끊겨도 `AUTO_RESUME=True`면 같은 output_dir에서 이어서 학습합니다.


In [ ]:
import subprocess, shlex, os
from pathlib import Path

def add_arg(cmd, key, value):
    if value is None:
        return
    if isinstance(value, bool):
        if value:
            cmd.append(key)
        return
    if isinstance(value, (list, tuple)):
        if len(value) > 0:
            cmd.append(key)
            cmd.extend([str(v) for v in value])
        return
    cmd.extend([key, str(value)])

split_cmd = [
    "python", "/content/recycling_vqa_colab_train_v2.py",
    "--mode", "train",
    "--preset", MODEL_PRESET,
    "--train_csv", train_csv_for_run,
    "--image_root", IMAGE_ROOT,
    "--image_col", IMAGE_COL,
    "--question_col", QUESTION_COL,
    "--option_cols", *OPTION_COLS,
    "--answer_col", ANSWER_COL,
    "--output_dir", SPLIT_OUTPUT_DIR,
    "--valid_ratio", str(VALID_RATIO),
    "--num_train_epochs", str(SEARCH_EPOCHS),
    "--lora_r", str(LORA_R),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--max_seq_length", str(MAX_SEQ_LENGTH),
    "--save_steps", str(SAVE_STEPS),
    "--eval_steps", str(EVAL_STEPS),
    "--logging_steps", str(LOGGING_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--seed", str(SEED),
]

add_arg(split_cmd, "--valid_csv", valid_csv_for_run)
add_arg(split_cmd, "--extra_train_csv", EXTRA_TRAIN_CSV)
add_arg(split_cmd, "--category_col", CATEGORY_COL)
add_arg(split_cmd, "--aux_image_cols", AUX_IMAGE_COLS)
add_arg(split_cmd, "--detector_hint_col", DETECTOR_HINT_COL)
add_arg(split_cmd, "--model_name", CUSTOM_MODEL_NAME)
add_arg(split_cmd, "--learning_rate", LEARNING_RATE)
add_arg(split_cmd, "--per_device_train_batch_size", TRAIN_BATCH_SIZE)
add_arg(split_cmd, "--gradient_accumulation_steps", GRAD_ACCUM)

if REMOVE_IDENTICAL_OPTION_ROWS:
    split_cmd.append("--remove_identical_option_rows")
if ENABLE_AUGMENTATION:
    split_cmd.append("--enable_augmentation")
if AUTO_RESUME:
    split_cmd.append("--auto_resume")
if RUN_EXACT_MATCH_EVAL:
    split_cmd.append("--run_exact_match_eval")
if EVAL_MAX_SAMPLES is not None:
    split_cmd.extend(["--eval_max_samples", str(EVAL_MAX_SAMPLES)])

print(" ".join(shlex.quote(x) for x in split_cmd))
subprocess.run(split_cmd, check=True)

print("[DONE] Split training finished.")


## 7) split 학습 결과 확인

- `run_config.json`: 이번 실행 설정
- `train_metrics.json`: trainer 지표
- `exact_match_eval.json`: 생성 기반 exact-match 결과
- `checkpoint-*`: 중간 저장 체크포인트


In [ ]:
from pathlib import Path
import json
import pandas as pd

out_dir = Path(SPLIT_OUTPUT_DIR)
print("Output dir:", out_dir)

if out_dir.exists():
    ckpts = sorted(out_dir.glob("checkpoint-*"))
    print("Checkpoints:", [p.name for p in ckpts][-5:])

    for name in ["run_config.json", "train_metrics.json", "exact_match_eval.json"]:
        path = out_dir / name
        if path.exists():
            print(f"\n--- {name} ---")
            print(path.read_text()[:2000])
else:
    print("[WARN] output dir not found")


## 8) 2차 full-data 2 epoch 학습

파라미터가 괜찮다고 판단되면 이 셀을 실행하세요.

포인트:
- `valid_ratio=0.0`
- `FULL_TRAIN_EPOCHS=2.0`
- 보통 최종 제출용 adapter는 여기서 만듭니다.
- 탐색이 끝난 뒤 32B로 바꾸고 다시 한 번 돌릴 때도 이 셀을 씁니다.


In [ ]:
import subprocess, shlex

full_cmd = [
    "python", "/content/recycling_vqa_colab_train_v2.py",
    "--mode", "train",
    "--preset", MODEL_PRESET,
    "--train_csv", TRAIN_CSV,
    "--image_root", IMAGE_ROOT,
    "--image_col", IMAGE_COL,
    "--question_col", QUESTION_COL,
    "--option_cols", *OPTION_COLS,
    "--answer_col", ANSWER_COL,
    "--output_dir", FULL_OUTPUT_DIR,
    "--valid_ratio", "0.0",
    "--num_train_epochs", str(FULL_TRAIN_EPOCHS),
    "--lora_r", str(LORA_R),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--max_seq_length", str(MAX_SEQ_LENGTH),
    "--save_steps", str(SAVE_STEPS),
    "--logging_steps", str(LOGGING_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--seed", str(SEED),
]

add_arg(full_cmd, "--extra_train_csv", EXTRA_TRAIN_CSV)
add_arg(full_cmd, "--category_col", CATEGORY_COL)
add_arg(full_cmd, "--aux_image_cols", AUX_IMAGE_COLS)
add_arg(full_cmd, "--detector_hint_col", DETECTOR_HINT_COL)
add_arg(full_cmd, "--model_name", CUSTOM_MODEL_NAME)
add_arg(full_cmd, "--learning_rate", LEARNING_RATE)
add_arg(full_cmd, "--per_device_train_batch_size", TRAIN_BATCH_SIZE)
add_arg(full_cmd, "--gradient_accumulation_steps", GRAD_ACCUM)

if REMOVE_IDENTICAL_OPTION_ROWS:
    full_cmd.append("--remove_identical_option_rows")
if ENABLE_AUGMENTATION:
    full_cmd.append("--enable_augmentation")
if AUTO_RESUME:
    full_cmd.append("--auto_resume")

print(" ".join(shlex.quote(x) for x in full_cmd))
# 최종 학습을 시작하려면 아래 주석을 해제하세요.
# subprocess.run(full_cmd, check=True)
print("[INFO] Ready for full-data training. Uncomment subprocess.run(...) when you want to launch it.")


## 9) 예측 / 제출 파일 생성

`PREDICT_FROM = "split"`이면 split run adapter,
`"full"`이면 full-data final adapter를 사용합니다.


In [ ]:
import subprocess, shlex
from pathlib import Path

adapter_dir = Path(SPLIT_OUTPUT_DIR) / "final_adapter"
if PREDICT_FROM == "full":
    adapter_dir = Path(FULL_OUTPUT_DIR) / "final_adapter"

pred_cmd = [
    "python", "/content/recycling_vqa_colab_train_v2.py",
    "--mode", "predict",
    "--preset", MODEL_PRESET,
    "--test_csv", TEST_CSV,
    "--image_root", IMAGE_ROOT,
    "--image_col", IMAGE_COL,
    "--question_col", QUESTION_COL,
    "--option_cols", *OPTION_COLS,
    "--prediction_output", PREDICTION_OUTPUT,
    "--adapter_path", str(adapter_dir),
    "--max_seq_length", str(MAX_SEQ_LENGTH),
    "--pred_max_new_tokens", str(PRED_MAX_NEW_TOKENS),
    "--seed", str(SEED),
]

add_arg(pred_cmd, "--category_col", CATEGORY_COL)
add_arg(pred_cmd, "--aux_image_cols", AUX_IMAGE_COLS)
add_arg(pred_cmd, "--detector_hint_col", DETECTOR_HINT_COL)
add_arg(pred_cmd, "--model_name", CUSTOM_MODEL_NAME)

print(" ".join(shlex.quote(x) for x in pred_cmd))
# 예측을 시작하려면 아래 주석을 해제하세요.
# subprocess.run(pred_cmd, check=True)
print("[INFO] Ready for prediction. Uncomment subprocess.run(...) when adapter가 준비되면 실행하세요.")


## 10) 제출 파일 확인


In [ ]:
from pathlib import Path
import pandas as pd

sub_path = Path(PREDICTION_OUTPUT)
if sub_path.exists():
    sub = pd.read_csv(sub_path)
    display(sub.head())
    print("rows:", len(sub))
    print("saved:", sub_path)
else:
    print("[INFO] submission.csv가 아직 없습니다. 예측 셀 실행 후 다시 확인하세요.")


## 11) detector crop을 붙일 때

현재 노트북은 detector-assisted multi-image 입력을 지원합니다.

예시:
- 원본 이미지 컬럼: `path`
- detector crop 컬럼: `roi_path`
- OCR crop 컬럼: `ocr_crop_path`

그럼 설정 셀에서 아래처럼 바꾸면 됩니다.

```python
AUX_IMAGE_COLS = ["roi_path", "ocr_crop_path"]
```

또 detector가 요약 텍스트를 준다면:

```python
DETECTOR_HINT_COL = "detector_hint"
```

이렇게 하면 원본 + crop + detector text hint를 함께 넣을 수 있습니다.
